# Week 7 — CTEs and Advanced Analytics: Advanced Analytics (DeepSeek Guided)
## Phase 2b SQL | PORA Academy Cohort 7 — **Demo**

By the end of this session, you will be able to:
- Combine CTEs with window functions (`LAG`, `RANK`, `NTILE`) to answer multi-step business analytics questions
- Use `LAG()` over an ordered result to compare each row against the one before it — the month-over-month growth pattern
- Use `RANK()` and a single-row "total" CTE together to turn raw revenue into a league table with each row's share of the whole
- Work the **draft → run → verify** DeepSeek protocol on queries whose answers you do *not* already know

### Setup — run this cell first

Same setup cell as yesterday. It loads all 8 Olist tables into a file-based SQLite database and
connects the `%%sql` magic to it, so every query below returns a pandas DataFrame. Run it before
anything else — nothing in this notebook works without it.

In [ ]:
# =====================================================================
# Olist SQL Setup — runs on BOTH Google Colab and a local machine.
# Run this cell FIRST. It loads the 8 Olist tables into a SQLite
# database and connects the %%sql magic to it. You should not need to
# edit anything unless auto-detection fails (see the two knobs below).
#
# Design notes:
# - We teach SQL with the %%sql cell magic (jupysql), not pd.read_sql().
# - jupysql opens its OWN connection, so the DB must be a real FILE
#   (a :memory: DB would be invisible to it).
# - We use jupysql (the maintained SQL magic). On Colab we install it,
#   because Colab ships the legacy ipython-sql, which (a) can't take a
#   connection by engine variable and (b) renders every result through
#   prettytable.__dict__[style], crashing on modern prettytable with
#   KeyError 'DEFAULT'/'SINGLE_BORDER'. jupysql fixes both.
# - autopandas=True makes every %%sql result a pandas DataFrame, which
#   lets the self-check cells assert on .iloc/.shape directly.
# =====================================================================
import os, glob, sqlite3, tempfile, zipfile
import pandas as pd

# --- Optional knobs (leave blank; only set if auto-detect fails) ------
LOCAL_DATA_DIR = ""   # local run: folder that holds olist_orders_dataset.csv
DRIVE_ZIP_PATH = ""   # Colab: full path to phase-2-python-sql.zip in your Drive
# ---------------------------------------------------------------------

# Detect Colab (google.colab only imports there). Outside Colab — including
# the content-pipeline validator — this falls through to the local branch.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ON_COLAB = True
except ModuleNotFoundError:
    ON_COLAB = False


def _colab_find_zip():
    """Locate phase-2-python-sql.zip in Drive WITHOUT a full recursive scan
    (globbing '/content/drive/MyDrive/**' walks the entire Drive over the
    network and can hang for many minutes). Try explicit paths first, then a
    depth- and count-bounded breadth-first search that prints progress."""
    if DRIVE_ZIP_PATH:
        if os.path.exists(DRIVE_ZIP_PATH):
            return DRIVE_ZIP_PATH
        raise FileNotFoundError(f"DRIVE_ZIP_PATH is set but not found: {DRIVE_ZIP_PATH}")

    target = "phase-2-python-sql.zip"
    # Fast, instant checks of the most likely spots (top of Drive + course folder).
    for cand in (
        f"/content/drive/MyDrive/{target}",
        f"/content/drive/MyDrive/Data Analysis and AI Automation Course Cohort 7/Dataset/{target}",
        f"/content/{target}",
    ):
        if os.path.exists(cand):
            return cand

    # Bounded BFS: depth <= 4, at most ~600 folders, skipping hidden dirs.
    print("Searching your Google Drive for phase-2-python-sql.zip ...")
    root, queue, scanned = "/content/drive/MyDrive", [("/content/drive/MyDrive", 0)], 0
    while queue:
        d, depth = queue.pop(0)
        hit = os.path.join(d, target)
        if os.path.exists(hit):
            return hit
        if depth >= 4:
            continue
        try:
            for e in os.scandir(d):
                if e.is_dir() and not e.name.startswith("."):
                    queue.append((e.path, depth + 1))
        except OSError:
            continue
        scanned += 1
        if scanned % 50 == 0:
            print(f"  ...scanned {scanned} folders")
        if scanned >= 600:
            break

    raise FileNotFoundError(
        "Could not quickly find phase-2-python-sql.zip in your Drive. Put the zip at the "
        "TOP of your Drive (My Drive) and re-run, or set DRIVE_ZIP_PATH at the top of this "
        "cell to its exact path.")


def _find_csv_dir():
    """Return the folder that actually contains olist_orders_dataset.csv."""
    roots = []
    env_dir = os.environ.get("OLIST_DATA_PATH", "")   # set by the pipeline validator
    if env_dir:
        roots.append(env_dir)
    if LOCAL_DATA_DIR:
        roots.append(LOCAL_DATA_DIR)

    if ON_COLAB:
        extract_path = "/content/olist_data"
        # unzip only the first time; reuse the extracted CSVs afterwards
        if not glob.glob(f"{extract_path}/**/olist_orders_dataset.csv", recursive=True):
            zip_path = _colab_find_zip()
            os.makedirs(extract_path, exist_ok=True)
            print(f"Unzipping {os.path.basename(zip_path)} ...")
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(extract_path)
        roots.append(extract_path)
    else:
        # Local: search cwd (recursively) + a few common spots — never the whole
        # home dir (that recursive walk can be very slow). Set LOCAL_DATA_DIR if
        # your CSVs live elsewhere.
        roots += [os.getcwd(),
                  os.path.expanduser("~/Downloads"),
                  os.path.expanduser("~/Desktop"),
                  os.path.expanduser("~/olist")]

    for root in roots:
        if os.path.exists(os.path.join(root, "olist_orders_dataset.csv")):
            return root
        hits = glob.glob(os.path.join(root, "**", "olist_orders_dataset.csv"), recursive=True)
        if hits:
            return os.path.dirname(hits[0])

    raise FileNotFoundError(
        "Olist CSVs not found. Set LOCAL_DATA_DIR (local) or DRIVE_ZIP_PATH (Colab) at "
        "the top of this cell.")


DATA_DIR = _find_csv_dir()
print("Data folder:", DATA_DIR)

# Build a file-based SQLite DB shared by pandas (loading) and jupysql (querying).
DB_PATH = os.environ.get("OLIST_DB_PATH") or (
    "/content/olist.db" if ON_COLAB else os.path.join(tempfile.gettempdir(), "olist.db"))

tables = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv",
}

conn = sqlite3.connect(DB_PATH)
for table_name, filename in tables.items():
    df = pd.read_csv(os.path.join(DATA_DIR, filename))
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"Loaded {table_name}: {len(df):,} rows")
conn.close()
print("\nDatabase ready.")

# On Colab, install jupysql so `%load_ext sql` loads it instead of the legacy
# ipython-sql (see header). Off Colab (local / pipeline validator) jupysql is
# already installed, so we skip the install and stay offline-safe.
if ON_COLAB:
    get_ipython().run_line_magic("pip", "install --quiet --upgrade jupysql")

get_ipython().run_line_magic("load_ext", "sql")

# Guard: if the legacy ipython-sql was already loaded earlier THIS session (e.g.
# an older cell ran first), the freshly installed jupysql cannot hot-swap in — a
# runtime restart is the only fix. jupysql exposes sql.connection.ConnectionManager;
# ipython-sql does not. Stop with a clear instruction instead of a later cryptic
# prettytable KeyError.
import sql.connection as _sqlconn
if not hasattr(_sqlconn, "ConnectionManager"):
    raise RuntimeError(
        "Legacy ipython-sql is active, not jupysql. On Colab: Runtime -> Restart session, "
        "then run THIS setup cell first (before any other cell). Locally: "
        "pip install --upgrade jupysql and restart the kernel."
    )

# Connect the %%sql magic to the SAME database file. autopandas=True is REQUIRED
# (see header). We connect with run_line_magic (not a literal `%sql` line) so the
# computed DB_PATH is interpolated correctly. Do NOT set SqlMagic.style.
get_ipython().run_line_magic("config", "SqlMagic.autopandas = True")
get_ipython().run_line_magic("config", "SqlMagic.feedback = 0")
get_ipython().run_line_magic("sql", f"sqlite:///{DB_PATH}")

# Verify (expected row counts — do not alter without re-running against data):
#   orders 99,441 | customers 99,441 | order_items 112,650 | order_payments 103,886
#   order_reviews 99,224 | products 32,951 | sellers 3,095 | product_category_translation 71

## Why this matters

Yesterday you learned to name your steps. Today you get a new kind of column to put inside them.

Here is the gap. Every query you have written so far answers a question about **one row at a time**,
or about a **group of rows collapsed into one**. Neither can answer the two questions Olist's
leadership actually asks at the end of a quarter. *"Are we growing?"* is not a question about a
month — it is a question about a month **compared to the month before it**. *"Which categories
matter?"* is not a question about a category's revenue — it is a question about that revenue
**relative to every other category and to the platform total**. `GROUP BY` throws away exactly the
context you need: once it has collapsed October into a single row, that row has no idea September
exists.

**Window functions** are the fix. A window function runs *after* the grouping, and it lets each
surviving row look sideways at its neighbours without collapsing them. `LAG()` reaches back to the
previous row. `RANK()` numbers rows by position. `NTILE()` cuts them into equal buckets. Paired with
the CTEs you learned yesterday — step one produces the rows, step two adds the sideways column, the
outer query reports — they cover most of what a working analyst is asked for.

You have real context for this. Olist's order volume went from **329 orders in 2016** to **45,101 in
2017** to **54,011 in 2018**, and November 2017 alone carried **7,544 orders** on the back of Black
Friday. Those totals tell you the platform grew; they cannot tell you *when* it grew, or which month
stalled. Today's first query can.

> 🤖 **This is a DeepSeek-guided session.** Unlike the queries you have verified all term, today's
> two business questions have no answer printed in your curriculum — you run them and read what
> comes back. That is exactly the situation where an AI assistant is most useful and most dangerous,
> so the "Using DeepSeek" section below is the centre of today's session, not a footnote.

## 1. `LAG()` — letting a row see the row before it

`LAG(column) OVER (ORDER BY something)` gives the current row the value that `column` holds in the
**previous** row, where "previous" means previous *in the order you specify inside the `OVER`
clause*. It adds a column; it never removes a row. That is the whole idea, and it is what separates a
window function from an aggregate: `SUM()` turns twelve rows into one, `LAG()` leaves all twelve
standing and hands each of them a piece of its neighbour.

The everyday analogy is a bank statement. Each line shows a balance, but what you actually care
about is the *change* since the line above. Your eye does the `LAG()` automatically — it reads this
row, glances up one row, and subtracts. A window function is you writing that glance down as SQL.

Two details decide whether the answer is right. First, **the `ORDER BY` inside `OVER (...)` is not
decoration** — it defines what "previous" means. Order by month and the previous row is last month;
order by order count and the previous row is whichever month happened to sell slightly less, which
is meaningless. Second, **the very first row has no previous row**, so `LAG()` returns `NULL` there.
That is correct behaviour, not a bug: there is genuinely nothing to compare January against. The
query below handles it in the outer step with `WHERE prev_month_count IS NOT NULL` — the
`IS NOT NULL` discipline from Week 1, still earning its keep.

The business question: **for each month, what was the growth rate in orders compared to the previous
month?** Three named steps — count the orders per month, attach the previous month's count, then
compute the percentage change.

In [ ]:
%%sql
-- Question 1: month-over-month growth in orders, 2017-2018.
-- Step 1 — one row per month, with that month's order count:
WITH monthly_orders AS (
    SELECT strftime('%Y-%m', order_purchase_timestamp) AS month,
           COUNT(*) AS order_count
    FROM orders
    WHERE strftime('%Y', order_purchase_timestamp) IN ('2017', '2018')
    GROUP BY month
),
-- Step 2 — reads step 1 and adds a sideways look at the row before it:
with_prev AS (
    SELECT month, order_count,
           LAG(order_count) OVER (ORDER BY month) AS prev_month_count
    FROM monthly_orders
)
-- Step 3 — the outer query turns the pair of numbers into a growth rate:
SELECT month,
       order_count,
       prev_month_count,
       ROUND((order_count - prev_month_count) * 100.0 / prev_month_count, 1) AS growth_pct
FROM with_prev
WHERE prev_month_count IS NOT NULL   -- drops the first month, which has no previous row
ORDER BY month

### Reading the result

Run it and read the `growth_pct` column top to bottom before you read anything else. There is no
expected table printed for this one — this is a genuine analysis, not a value to match — so the
skill being practised is interpretation.

Three things to look for:

- **Where the sign flips.** Any negative number is a month that sold less than the month before it.
  One negative month in a growing business is noise; two or three in a row is a trend, and a trend
  is what a leadership team wants explained.
- **What happens around November 2017.** You already know that month carried 7,544 orders on Black
  Friday — the single biggest month in the dataset. A spike like that shows up **twice** in a
  month-over-month column: once as the jump into it, and once as the fall out of it the following
  month. The December drop is not a decline in the business; it is the Black Friday spike ending.
  This is the classic trap of month-over-month reporting, and the reason analysts pair it with
  year-over-year.
- **Where the series stops.** The dataset's order history ends partway through 2018, so the final
  months are thin. Do not read a collapse into a month that simply has not finished.

Now look at the SQL itself, because two lines in it are load-bearing:

`* 100.0` — `order_count` and `prev_month_count` are both **integers**. Written as `* 100`, SQLite
would do integer division and a real 12% growth would print as `12`… or a real 0.8% would print as
`0`. The `.0` forces REAL arithmetic. This is the Week 2 trap in its most expensive form, because
nothing errors — you just quietly report the wrong growth.

`ORDER BY month` appears **twice**, and the two do different jobs. The one inside `OVER (...)` tells
`LAG()` which row counts as "previous" and is essential to correctness. The one at the bottom sorts
the rows you finally display, and is cosmetic. Deleting the inner one does not error — it just
returns nonsense. Never leave it out.

One more property worth naming: `strftime('%Y-%m', ...)` produces strings like `'2017-03'`, and
sorting those **as text** puts them in true chronological order, because the format is
zero-padded and largest-unit-first. That is not luck — it is why this format is the standard for
date keys. Had you written `'%m-%Y'`, the text sort would order every March before every April
regardless of year, and `LAG()` would compare across years.

## 2. `RANK()` + share of total — a category league table

Second business question: **what percentage of total platform revenue does each product category
represent?** This is the shape of question that becomes a slide in every board deck: a ranked list,
each row carrying both an absolute number and its share of the whole.

It needs two things a single `GROUP BY` cannot give you. The first is the **grand total** — but
`SUM(price)` inside a query already grouped by category gives you the category's total, not the
platform's, and you cannot use one `GROUP BY` at two different grains in the same step. The second
is a **rank number** — a 1, 2, 3 column, not just a sort order, so the row can say "we are third"
even after someone copies it into a spreadsheet and re-sorts it.

Yesterday's lesson solves the first: define a second CTE that reads the first one and collapses it
to a single row holding the grand total. A one-row, one-column CTE like that is called a **scalar
CTE**, and the outer query can drop it into any expression as `(SELECT total_revenue FROM total)`.

Before we chain anything, run step one on its own — the habit from yesterday. This is the
category-revenue block, and it is the only part of today's second query with a number you can check
against the verified stats.

In [ ]:
%%sql
-- Step 1 on its own: revenue per (English) category — what the CTE will hand to the next step.
-- order_items is the only many-rows-per-order table here, so SUM(price) is at the right grain.
SELECT t.product_category_name_english AS category,
       ROUND(SUM(oi.price), 2) AS revenue
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
JOIN product_category_translation t ON p.product_category_name = t.product_category_name
GROUP BY category
ORDER BY revenue DESC
LIMIT 5
-- Expected top row: health_beauty = R$1,258,681.34

`health_beauty` at **R$1,258,681.34** is the verified top category from your reference stats, so
step one is producing the right numbers at the right grain. Anchor on that row — it is the one
number in today's second query that you can check, and it is what you will use in a minute to
verify anything DeepSeek writes for you.

Now add the two new pieces:

**The scalar CTE.** `total` reads `category_revenue` and sums it down to a single row with a single
column. The outer query then divides each category's revenue by `(SELECT total_revenue FROM total)`.
Because that subquery returns exactly one value, SQLite is happy to use it anywhere a number is
allowed. Note again the `* 100.0` before the division — habit, always.

**`RANK() OVER (ORDER BY revenue DESC)`.** This numbers the rows 1, 2, 3 … by revenue, highest
first, and writes that number into a real column. It does *not* sort your output — the `ORDER BY` at
the bottom of the query still does that — it only assigns the number. The `ORDER BY` inside `OVER`
and the `ORDER BY` at the end are separate instructions that happen to agree here; when they
disagree, you get a correctly-ranked table displayed in some other order, which is occasionally
exactly what you want.

One thing to know about `RANK()` specifically: on ties it gives both rows the same number and then
**skips** — two rows tied at 2 are followed by rank 4, not 3. `DENSE_RANK()` would give 2, 2, 3, and
`ROW_NUMBER()` would break the tie arbitrarily and give 2, 3. With revenue measured to the cent,
ties are vanishingly unlikely here, but on review scores or item counts they are routine. Pick the
one whose tie behaviour you actually want.

In [ ]:
%%sql
-- Question 2: each category's revenue, its share of the total, and its rank.
-- Step 1 — revenue per category:
WITH category_revenue AS (
    SELECT t.product_category_name_english AS category,
           ROUND(SUM(oi.price), 2) AS revenue
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    JOIN product_category_translation t ON p.product_category_name = t.product_category_name
    GROUP BY category
),
-- Step 2 — a SCALAR CTE: one row, one column, the grand total to divide by:
total AS (
    SELECT SUM(revenue) AS total_revenue FROM category_revenue
)
-- Step 3 — share of total (note the * 100.0) and a real rank column:
SELECT category,
       revenue,
       ROUND(revenue * 100.0 / (SELECT total_revenue FROM total), 2) AS revenue_pct,
       RANK() OVER (ORDER BY revenue DESC) AS rank_num
FROM category_revenue
ORDER BY revenue DESC
LIMIT 10

### Reading the result — and one honest caveat about the denominator

Again, no expected table: run it, then read it as an analyst would. Look at how quickly
`revenue_pct` falls away as you go down the list, and add the first few together in your head. The
useful business question is not "which category is biggest" — it is **how many categories you need
before you have covered half the platform's revenue**. That number decides whether Olist is a
marketplace with a handful of load-bearing categories or a genuinely broad one, and it changes how
you would staff category management.

Now the caveat, and it is the kind of thing that separates a careful analyst from a confident one.
**Ask what the denominator actually is.** `total_revenue` is the sum of `category_revenue` — which
is the sum over items whose product has a category *and* whose category has an English
translation. It is therefore **not** the platform's full product revenue of R$13,591,643.70, which
includes items on the 610 products with a `NULL` category and any category the translation table
does not cover. The percentages in this table are shares of *translated-category revenue*, and they
are slightly larger than the same categories' shares of true platform revenue.

That is not a bug — the query answers exactly what it was asked — but it is the label you must put
on the slide. "Share of categorised revenue" and "share of platform revenue" are different claims,
and only one of them is what this query computes. If you needed the second, you would build the
denominator from a `SELECT SUM(price) FROM order_items` CTE instead, with no join to `products` at
all, and the percentages would all move down a little.

Last thing worth noticing: because `category_revenue` is read by **both** `total` and the outer
query, changing the filter or grain in that one block updates the numerator and the denominator
together. That is yesterday's "define the step once, reuse it" property doing real safety work — a
hand-pasted denominator is how a report ends up with percentages that do not sum to 100.

---
## 🤖 Using DeepSeek — the centre of today's session

Today is billed as **DeepSeek Guided**, and here is why that matters more today than it did in
Weeks 4, 5 or 6. Every session so far handed you the answer: the curriculum printed 96,478 delivered
orders, or R$154.10 average payment, and you checked the AI's SQL against it. **Today's two business
questions have no printed answer.** You are in the situation a working analyst is in every day —
asking a question nobody has answered yet, with a tool that will answer it confidently whether it is
right or wrong.

Window functions make this sharper still, because their failure mode is silent. An AI-drafted
`LAG()` query with the wrong `ORDER BY` inside `OVER (...)` runs perfectly and returns a full,
plausible table of growth percentages that are simply comparing the wrong pairs of months. Nothing
is red. Nothing is empty. The numbers are just wrong, and the only way you find out is by checking.

**The protocol — draft, run, verify — does not change. What changes is what you verify against.**

1. **Ask precisely, naming the steps.** Vague prompts get vague SQL. Try:
   *"Using SQLite, write a query with a CTE called `monthly_orders` that counts orders per month for
   2017 and 2018 using `strftime`, then a second CTE that adds the previous month's count with
   `LAG()` ordered by month, then an outer query returning the month-over-month growth percentage
   rounded to 1 decimal. Dates are stored as TEXT."*
   Telling it the dates are TEXT and that the engine is **SQLite** (not PostgreSQL or MySQL) prevents
   half the errors you would otherwise debug.
2. **Run it here, and read the `OVER` clauses out loud.** For every window function it writes, ask
   two questions: *what does `ORDER BY` inside this `OVER` say "previous" means?* and *does that
   match what the business question means?* If you cannot answer, ask DeepSeek to explain that clause
   specifically — not to rewrite the query.
3. **Verify against a number you already hold.** When the answer itself is unknown, you verify a
   *component* of it. You know `health_beauty` is R$1,258,681.34. You know 2017 had 45,101 orders and
   2018 had 54,011. You know November 2017 had 7,544. So make the AI's query reproduce one of those:
   filter it to a single category, or sum its monthly counts back up to a year, and check. If a
   component you can verify comes out wrong, every number in the table is wrong too — and if the
   components are right, you have earned the right to trust the parts you cannot check.

**Two rules that do not bend.** Never paste a number from an AI's *prose* into your work — only
numbers a query you ran actually returned; DeepSeek will happily write "health_beauty accounts for
roughly 9% of revenue" without having touched the data. And never let AI-drafted SQL skip the
fan-out check from last week: neither of today's queries joins two of `order_items`,
`order_payments` or `order_reviews`, but a "helpful" AI addition of review scores to the category
table would do exactly that and inflate every revenue figure on the slide.

The cell below is step 3 in practice — the component check you run against any AI-drafted version of
today's second query.

In [ ]:
%%sql
-- Step 3 of the protocol: force an AI-drafted query to reproduce a number we already trust.
-- If health_beauty does not come back at R$1,258,681.34, the QUERY is wrong -- not the data.
WITH category_revenue AS (
    SELECT t.product_category_name_english AS category,
           ROUND(SUM(oi.price), 2) AS revenue
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    JOIN product_category_translation t ON p.product_category_name = t.product_category_name
    GROUP BY category
)
SELECT category, revenue
FROM category_revenue
WHERE category = 'health_beauty'
-- Expected: health_beauty | 1258681.34

## Going deeper — `NTILE()`, when a rank is too precise

`RANK()` answers "which position is this row in?". Sometimes that is more precision than the
business can use. Nobody runs a different strategy for the 47th-best category than for the 48th —
but they absolutely run a different strategy for the **top quarter** than for the bottom quarter.

`NTILE(n) OVER (ORDER BY ...)` sorts the rows and cuts them into `n` buckets of (as near as
possible) equal **size**, labelling each row 1 to `n`. `NTILE(4)` gives quartiles, `NTILE(10)`
deciles, `NTILE(2)` halves. Yesterday you produced tiers with `CASE WHEN total_revenue >= 100000`,
and it is worth being clear about how the two differ, because they answer different questions:

- **`CASE` thresholds** cut by **value**. The bands mean something fixed — "R$100k+" — and they can
  come out wildly unbalanced (yesterday: 18 sellers in one band, 2,803 in another). Use them when
  the threshold itself is the business rule.
- **`NTILE`** cuts by **position**. Every bucket holds the same number of rows by construction, and
  what counts as "top quarter" moves as the data moves. Use it when you want a fixed *share* of the
  population in each band — the top 25% of anything.

The query below cuts product categories into revenue quartiles and reports what each quartile
contributes. Read the bucket sizes first (they should be near-equal — that is what `NTILE` is for),
then read the revenue totals beside them, which will not be equal at all. The gap between those two
columns *is* the concentration story.

One caution: `NTILE` can only split rows into whole buckets, so when the row count does not divide
evenly the earlier buckets get one extra row. And like every window function, it is meaningless
without the `ORDER BY` inside `OVER (...)` — without it, your "quartiles" are four arbitrary piles.

In [ ]:
%%sql
-- NTILE(4): cut categories into four equal-SIZED buckets by revenue, then summarise each bucket.
WITH category_revenue AS (
    SELECT t.product_category_name_english AS category,
           ROUND(SUM(oi.price), 2) AS revenue
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    JOIN product_category_translation t ON p.product_category_name = t.product_category_name
    GROUP BY category
),
quartiles AS (
    SELECT category,
           revenue,
           NTILE(4) OVER (ORDER BY revenue DESC) AS revenue_quartile
    FROM category_revenue
)
SELECT revenue_quartile,
       COUNT(*) AS category_count,          -- near-equal by construction: that's what NTILE does
       ROUND(SUM(revenue), 2) AS quartile_revenue,
       ROUND(MIN(revenue), 2) AS smallest_in_quartile,
       ROUND(MAX(revenue), 2) AS largest_in_quartile
FROM quartiles
GROUP BY revenue_quartile
ORDER BY revenue_quartile

## Common mistakes

**Mistake — filtering on a window function in the same `SELECT` that creates it.** This is the big
one, and it is why today's first query has three steps instead of two. `WHERE` runs *before* window
functions are computed, so the column does not exist yet when `WHERE` looks for it. Writing
`... LAG(order_count) OVER (ORDER BY month) AS prev_month_count ... WHERE prev_month_count IS NOT
NULL` in one `SELECT` fails with `no such column: prev_month_count`, and trying
`WHERE LAG(order_count) OVER (ORDER BY month) IS NOT NULL` fails with `misuse of window function`.
**The fix is always the same: compute the window column inside a CTE, then filter in the outer
query** — exactly what `with_prev` exists to do.

**Mistake — omitting `ORDER BY` inside `OVER (...)`.** `LAG(order_count) OVER ()` runs without
complaint and returns "the previous row" in whatever order SQLite happened to produce rows. Your
growth percentages will be real numbers computed from arbitrary pairs. Nothing warns you. If a
window function's meaning depends on sequence — and `LAG`, `RANK` and `NTILE` all do — the
`ORDER BY` goes **inside the `OVER`**, and the one at the end of the query is a separate, cosmetic
thing.

**Mistake — integer division in the growth percentage.** `(order_count - prev_month_count) * 100 /
prev_month_count` on two integer columns truncates: 12.7% prints as `12`, and 0.8% prints as `0`.
Always `* 100.0`.

**Mistake — `= NULL` on the first row.** The month with no predecessor comes back as `NULL`, and
`WHERE prev_month_count != NULL` matches **zero rows**, silently returning an empty table. `NULL` is
never equal or unequal to anything; use `IS NOT NULL`.

**Mistake — assuming `RANK()` has no gaps.** Ties share a rank and the next rank skips ahead (1, 2,
2, 4). If your league table must number 1..n with no gaps, you want `DENSE_RANK()` or
`ROW_NUMBER()`.

The cell below shows the first two as comments, then runs the correct version live — the same
`LAG()` pipeline, this time keeping only the months that **shrank**.

In [ ]:
%%sql
-- ── COMMON MISTAKES ─────────────────────────────────────────────────
-- WRONG #1 — filtering on a window column in the SELECT that creates it:
--   SELECT month, LAG(order_count) OVER (ORDER BY month) AS prev_month_count
--   FROM monthly_orders
--   WHERE prev_month_count IS NOT NULL
--   -> no such column: prev_month_count   (WHERE runs BEFORE window functions)
--   ...and moving the function into the WHERE is worse:
--   WHERE LAG(order_count) OVER (ORDER BY month) IS NOT NULL
--   -> misuse of window function LAG()
--
-- WRONG #2 — no ORDER BY inside OVER (); runs fine, compares arbitrary rows:
--   LAG(order_count) OVER () AS prev_month_count
--
-- CORRECT — window column built in a CTE, filtered in the OUTER query.
-- Same pipeline as Question 1, narrowed to the months that SHRANK:
WITH monthly_orders AS (
    SELECT strftime('%Y-%m', order_purchase_timestamp) AS month,
           COUNT(*) AS order_count
    FROM orders
    WHERE strftime('%Y', order_purchase_timestamp) IN ('2017', '2018')
    GROUP BY month
),
with_prev AS (
    SELECT month, order_count,
           LAG(order_count) OVER (ORDER BY month) AS prev_month_count
    FROM monthly_orders
),
with_growth AS (
    SELECT month, order_count, prev_month_count,
           ROUND((order_count - prev_month_count) * 100.0 / prev_month_count, 1) AS growth_pct
    FROM with_prev
    WHERE prev_month_count IS NOT NULL     -- IS NOT NULL, never != NULL
)
SELECT month, order_count, prev_month_count, growth_pct
FROM with_growth
WHERE growth_pct < 0                        -- only the months that went backwards
ORDER BY growth_pct

## Group exercise — interrogate an AI's window function

⏱ ~8 min · pairs or threes · discussion only, no code required

Today's protocol is only as good as the questions you ask of a draft. Practise asking them out loud.
Nominate someone to report back one sentence per part.

**Part A — decide the grain before you write anything.** Tonight's exercises include *"using
`LAG()`, calculate month-over-month **revenue** growth using `order_payments`"*. Before any SQL:
agree on what the named steps are, and settle two questions between you. (1) Which column carries
the month — `order_payments` has no date, so where does the month come from, and what does that join
imply? (2) `order_payments` holds **103,886 rows for 99,440 distinct orders** because instalments and
vouchers are separate rows. Does that break a monthly `SUM(payment_value)`, or not — and why is your
answer different from what it would be if you were counting *orders* instead of summing *money*?

**Part B — ties, and which ranking function you actually want.** Suppose you rank the five review
scores by how many reviews each received, and two scores come back with an identical count. Work out
what `RANK()`, `DENSE_RANK()` and `ROW_NUMBER()` each return for the rows after the tie. Then pick
one for this scenario: *"list the top 3 categories by revenue in each state"*. Which function makes
"top 3" mean what the business means, and what goes wrong with the other two?

Be ready to defend Part A with an argument about **grain**, not a hunch.

## Mini-challenge — your turn

⏱ ~5–10 min

Take the Question 2 query — `category_revenue`, `total`, and the outer `SELECT` with `revenue_pct`
and `RANK()` — and return **only the `health_beauty` row**, without losing the ranking.

Two hints, and the second one is the actual lesson:

- Keep both CTEs exactly as they are. Nothing inside them changes.
- **Do not** just add `WHERE category = 'health_beauty'` next to the `RANK()`. `WHERE` runs before
  window functions, so the rank would then be computed over the single surviving row and come back
  as 1 no matter what the truth is. To keep a rank that means something, wrap the whole ranked query
  in one more CTE (call it `ranked`) and filter **outside** it. This is the same "compute inside,
  filter outside" shape as the mistakes section — and the reason it matters is that both the wrong
  and the right version run without error.

**Expected:** one row, with `revenue` reading **1258681.34** — the verified platform top category
from your reference stats. That is your anchor: if the revenue matches, your CTEs survived the edit
intact. Then compare the `rank_num` your version returns against what you get from the
filter-in-the-wrong-place version, and satisfy yourself you understand why they can differ.

**Stretch question to discuss:** if you wanted `revenue_pct` to be a share of *all* product revenue
(R$13,591,643.70) rather than of translated-category revenue, which single CTE would you rewrite —
and would `health_beauty`'s percentage go up or down?

In [ ]:
%%sql
-- ⏱ ~5-10 min — your turn! Replace the placeholder below with your own query.
SELECT 'write your query here' AS todo

## Session Summary

| Pattern | What it does | Example |
|---|---|---|
| `LAG(col) OVER (ORDER BY x)` | hands each row the previous row's value — no rows collapsed | `LAG(order_count) OVER (ORDER BY month) AS prev_month_count` |
| Window column in a CTE, filter outside | the only way to filter on a window result (`WHERE` runs first) | `WITH with_prev AS (SELECT ... LAG(...) ...) SELECT ... FROM with_prev WHERE prev_month_count IS NOT NULL` |
| `* 100.0` in a growth/share formula | forces REAL division so a percentage isn't truncated to an integer | `ROUND((order_count - prev_month_count) * 100.0 / prev_month_count, 1)` |
| Scalar CTE as a denominator | one row, one column — usable anywhere a number is allowed | `total AS (SELECT SUM(revenue) AS total_revenue FROM category_revenue)` |
| `RANK() OVER (ORDER BY x DESC)` | writes a real position column; ties share a rank and the next one skips | `RANK() OVER (ORDER BY revenue DESC) AS rank_num` |
| `NTILE(n) OVER (ORDER BY x)` | equal-**sized** buckets by position (vs `CASE`, which cuts by **value**) | `NTILE(4) OVER (ORDER BY revenue DESC) AS revenue_quartile` |

**What you can now answer that you couldn't yesterday:** how a month compares to the month before
it, what share of the whole each row of a summary represents, and where any row sits in a ranked
league table — all inside the same `WITH` pipeline you learned on Wednesday.

**The habit to take away:** with a window function, always ask *what does `ORDER BY` inside the
`OVER` mean here?* and *is this column being filtered in a step after the one that created it?*
Those two questions catch nearly every window-function bug, your own and DeepSeek's.

**And the DeepSeek habit:** when the answer is unknown, verify a **component** you do know —
`health_beauty` at R$1,258,681.34, 45,101 orders in 2017, 7,544 in November 2017. A query that gets
a known component wrong is wrong everywhere.

---
**Coming up next week — Week 8: End-to-End Business Analysis**, the capstone week of Phase 2b. No
new clauses: you'll take everything from Weeks 1–7 — filtering, aggregation, joins, `CASE`, dates,
subqueries, CTEs and today's window functions — and run a complete analysis of the Olist business
from question to finished result, the way you would be asked to at work. Come with this week's
assignment done; the `WITH` + window pipeline is the backbone of what you'll build.